# Distribution-fitting engine: AIC vs KS

Validates that the resource-productivity perspective identifies the *generating*
distribution family of each `(resource, task)` pair, comparing two selection
criteria: **AIC** and the **Kolmogorov-Smirnov** statistic. Reproduces Table 5.4 and
Figure 5.2 of the thesis (92 pairs; AIC 88.0 % vs KS 73.9 %).

## Prerequisite: generate the input logs

The logs are **not versioned** - they are generated with the scripts already in the
repository. Each script must be run **from its own directory**, because it uses
relative paths (`./modelo.bpmn`, `./prosimos_base.json`) and writes a temporary
`log.csv` in the working directory:

```bash
cd src/log_generation/resource_productivity/Caso_A
python generador_no_drift.py   # generates both the 2500- and 5000-trace logs
```

Repeat for `Caso_B`, `Caso_C` and `Caso_D`. This produces the 8 logs under
`data/01_raw/resource_productivity/Caso_*/` that the next cells read.

Each case probes a different property of the detector: **A** is the exploratory
baseline, **B** isolates identifiability (two distinguishable families per task),
**C** checks consistency with many heterogeneous pairs, and **D** is the control
(every resource uses the same generating family, so any discrepancy reveals the
detector's own noise).


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from pm4py.read import read_xes
import itertools

# Configurar pandas para mostrar todo sin recortes
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Agregar la raíz del proyecto al path
# En Jupyter no existe __file__, así que partimos del cwd y subimos hasta encontrar src/
project_root = Path(os.getcwd()).resolve()
while not (project_root / 'src').is_dir():
    if project_root.parent == project_root:
        raise RuntimeError("No se encontró la raíz del proyecto (directorio con 'src/')")
    project_root = project_root.parent
project_root = str(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.perspectivas.resource_profiles import filter_resource_productivity, transformacion_resource_productivity
import src.perspectivas.resource_profiles as _rp  # acceso al calendario singleton para resetearlo entre logs

In [ ]:
def _cargar_log(nombre_log: str) -> pd.DataFrame:
    
    ruta_log = Path(project_root) / "data" / "01_raw" / "resource_productivity" / nombre_log
    extension_log = ruta_log.suffix.lower()
    if extension_log == '.xes':
        log = read_xes(str(ruta_log))
    elif extension_log == '.csv':
        log = pd.read_csv(ruta_log)
    log['trace_real_index'] = (log['case:concept:name'] != log['case:concept:name'].shift(1)).cumsum()

    print(f"Log cargado: {ruta_log}")

    return log

In [ ]:
# Familias con soporte positivo: hay que fijar floc=0 para no inflar el ajuste.
# Para `norm` el loc es la media, así que no se fija.
#FAMILIAS_CANDIDATAS = [st.lognorm, st.expon, st.gamma, st.weibull_min, st.norm, st.uniform]
#FAMILIAS_SOPORTE_POSITIVO = {'lognorm', 'expon', 'gamma', 'weibull_min'}

FAMILIAS_CANDIDATAS = [st.lognorm, st.expon, st.gamma, st.norm, st.uniform]
FAMILIAS_SOPORTE_POSITIVO = {'lognorm', 'expon', 'gamma'}

def _encontrar_mejor_distribucion_KS(datos, recurso, tarea):
    """
    Variante de CONTROL del experimento: idéntica a `_encontrar_mejor_distribucion_AIC`
    (mismas familias candidatas, mismo ajuste por MLE con floc=0 en las familias de
    soporte positivo) salvo en el CRITERIO DE SELECCIÓN: aquí gana la familia con el
    menor estadístico de Kolmogorov-Smirnov (distancia máxima D entre la ECDF empírica
    y la CDF teórica), en lugar de la de menor AIC.

    Al compartir exactamente el mismo ajuste que la versión AIC, la única variable que
    cambia entre ambas es la regla de decisión. Esto permite atribuir cualquier
    diferencia de acierto al criterio en sí (distancia máxima vs verosimilitud
    penalizada por nº de parámetros), y no a diferencias en el preajuste.
    """

    mejor_familia  = None
    menor_error_ks = float('inf')

    for distribucion in FAMILIAS_CANDIDATAS:

        # Algunas familias (lognorm, gamma) tienen soporte estrictamente positivo o pueden
        # converger fuera del rango admisible (FitError). Si el ajuste falla, se omite la
        # familia y se sigue evaluando el resto.
        try:

            # A. Entrenar la familia con los datos (mismo ajuste que en la versión AIC).
            # Para familias con soporte positivo, fijamos loc=0 para evitar el
            # parámetro de desplazamiento artificial.
            if distribucion.name in FAMILIAS_SOPORTE_POSITIVO:
                parametros = distribucion.fit(datos, floc=0)
            else:
                parametros = distribucion.fit(datos)

            # B. Calcular la distancia KS (estadístico D): distancia vertical máxima
            # entre la ECDF de los datos y la CDF teórica de la familia ajustada.
            error_ks = st.kstest(datos, distribucion.cdf, args=parametros).statistic

        except Exception as e:
            print(f"Familia {distribucion.name:<12} | NO AJUSTABLE ({e.__class__.__name__})")
            continue

        print(f"Familia {distribucion.name:<12} | Error KS (D): {error_ks:.4f}")

        # C. Guardar la ganadora (distancia D más baja).
        if error_ks < menor_error_ks:
            menor_error_ks = error_ks
            mejor_familia  = distribucion

    if mejor_familia is None:
        print(f"Ninguna familia se ajusta a los datos de Recurso='{recurso}' y Tarea='{tarea}'.")
        return None

    print("-" * 39)
    print(f"LA FAMILIA GANADORA ES: '{mejor_familia.name}', recurso='{recurso}', tarea='{tarea}' con D KS={menor_error_ks:.4f}")

    return mejor_familia


def _encontrar_mejor_distribucion_AIC(datos, recurso, tarea):
    """
    Ajusta cada familia candidata por MLE y elige la mejor por AIC (no por KS).

    Dos cambios clave respecto a la versión anterior:
    - Para familias con soporte [0, inf) se fija floc=0 en el fit. Si se deja libre,
      el parámetro de localización absorbe sesgos y cualquier familia gana
      artificialmente (problema clásico al ajustar distribuciones desplazadas).
    - Selección por AIC en lugar de estadístico KS. El KS sobre los mismos datos
      con los que se ajustó la distribución (Lilliefors) está sesgado a la baja
      y favorece familias con más parámetros libres. AIC penaliza el número de
      parámetros y es el criterio correcto para *comparar* familias entre sí.
    """

    mejor_familia = None
    mejor_aic     = float('inf')

    for distribucion in FAMILIAS_CANDIDATAS:

        # Algunas familias (lognorm, gamma, weibull_min) tienen soporte estrictamente
        # positivo o pueden converger fuera del rango admisible (FitError). Si el ajuste
        # falla, se omite la familia y se sigue evaluando el resto.
        try:

            # A. Entrenar la familia con los datos.
            # Para familias con soporte positivo, fijamos loc=0 para evitar el
            # parámetro de desplazamiento artificial.
            if distribucion.name in FAMILIAS_SOPORTE_POSITIVO:
                parametros = distribucion.fit(datos, floc=0)
            else:
                parametros = distribucion.fit(datos)

            # B. Calcular el AIC del ajuste.
            # AIC = 2k - 2·log(L), donde k es el número de parámetros LIBRES
            # (los que realmente se han optimizado, no los que se han fijado).
            log_verosimilitud = np.sum(distribucion.logpdf(datos, *parametros))
            n_param_libres    = len(parametros) - (1 if distribucion.name in FAMILIAS_SOPORTE_POSITIVO else 0)
            aic               = 2 * n_param_libres - 2 * log_verosimilitud

        except Exception as e:
            print(f"Familia {distribucion.name:<12} | NO AJUSTABLE ({e.__class__.__name__})")
            continue

        print(f"Familia {distribucion.name:<12} | AIC: {aic:.2f}")

        # C. Guardar la ganadora (AIC más bajo).
        if aic < mejor_aic:
            mejor_aic     = aic
            mejor_familia = distribucion

    if mejor_familia is None:
        print(f"Ninguna familia se ajusta a los datos de Recurso='{recurso}' y Tarea='{tarea}'.")
        return None

    print("-" * 39)
    print(f"LA FAMILIA GANADORA ES: '{mejor_familia.name}', recurso='{recurso}', tarea='{tarea}' con AIC={mejor_aic:.2f}")

    return mejor_familia


In [ ]:
def _representar_resultados(resultados):

    distribuciones = [resultado[2] for resultado in resultados]

    # Diagrama de tarta de distribuciones ganadoras
    plt.figure(figsize=(8, 8))
    plt.pie(
        [distribuciones.count(dist) for dist in set(distribuciones)],
        labels=set(distribuciones),
        autopct='%1.1f%%',
        startangle=140,
        colors=sns.color_palette('pastel')
    )

In [ ]:
def _representar_distribucion_ganadora(datos, familia_evaluada):

    # Validación Visual
    # Comprobamos visualmente la correspondencia entre la distribución real de los datos 
    # y la forma teórica de la familia que estamos evaluando.

    # Ajustamos la familia elegida a nuestros datos
    if familia_evaluada.name in FAMILIAS_SOPORTE_POSITIVO:
        parametros = familia_evaluada.fit(datos, floc=0)
    else:
        parametros = familia_evaluada.fit(datos)

    fig, ejes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Análisis de Distribución contra Familia: {familia_evaluada.name.upper()}", fontsize=16)

    # --- GRÁFICO 1: Histograma + KDE ---
    # Analiza el primer gráfico (Histograma + KDE) para confirmar que la curva 
    # teórica abraza la forma general de la distribución real.
    sns.histplot(datos, kde=True, stat="density", ax=ejes[0], color='skyblue')
    # Dibujamos también la forma teórica para comparar visualmente
    x_plot = np.linspace(min(datos), max(datos), 100)
    ejes[0].plot(x_plot, familia_evaluada.pdf(x_plot, *parametros), color='red', linestyle='--', label='Teórica')
    ejes[0].set_title("1. Histograma + KDE")
    ejes[0].legend()

    # --- GRÁFICO 2: Q-Q Plot ---
    # Si los puntos caen sobre la diagonal → ajuste bueno; 
    # si se desvían en las colas → la familia no captura el comportamiento extremo
    st.probplot(datos, dist=familia_evaluada.name, sparams=parametros, plot=ejes[1])
    ejes[1].set_title("2. Q-Q Plot (El Detector en las Colas)")
    ejes[1].get_lines()[0].set_markerfacecolor('skyblue') # Color de los puntos
    ejes[1].get_lines()[0].set_markeredgecolor('black')

    # --- GRÁFICO 3: ECDF (Empírica) vs CDF (Teórica) ---
    sns.ecdfplot(datos, ax=ejes[2], label='ECDF (Datos Reales)', color='blue')
    ejes[2].plot(x_plot, familia_evaluada.cdf(x_plot, *parametros), color='red', linestyle='--', label='CDF (Teórica)')
    ejes[2].set_title("3. ECDF vs CDF Teórica")
    ejes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
LISTADO_LOGS = ["Caso_A/log_productivity_caso_a_2500.csv", "Caso_A/log_productivity_caso_a_5000.csv",
                "Caso_B/log_productivity_caso_b_2500.csv", "Caso_B/log_productivity_caso_b_5000.csv", 
                "Caso_C/log_productivity_caso_c_2500.csv", "Caso_C/log_productivity_caso_c_5000.csv",
                "Caso_D/log_productivity_caso_d_2500.csv", "Caso_D/log_productivity_caso_d_5000.csv",
                ]
RESULTADOS = []

for nombre_log in LISTADO_LOGS:
    # La transformación descubre un calendario y lo cachea como singleton de proceso
    # (optimización del pipeline real, que procesa un único log por ejecución). Aquí
    # recorremos varios logs en el mismo kernel, así que reseteamos el cache para que
    # cada log use SU calendario y no el del primero (Caso_A).
    _rp._obtener_calendario.calendario_estatico = None

    # Cargar el log
    log = _cargar_log(nombre_log)

    # Simular el filtrado y transformación del log para obtener los ratios de productividad
    log_filtrado = filter_resource_productivity(log, parametros={'debug': False})
    log_transformado = transformacion_resource_productivity(log_filtrado, parametros={'debug': False})

    print(log_transformado.head(10))

    # Motor de Búsqueda Automática de Distribuciones
    # Evaluamos varias familias de distribuciones para encontrar la que mejor se ajusta a nuestros datos
    
    # Extraemos los recursos y tareas únicos para generar combinaciones
    recursos = log_transformado['org:resource'].unique()
    tareas = log_transformado['concept:name'].unique()
        
    combinaciones = list(itertools.product(recursos, tareas))

    for recurso, tarea in combinaciones:

        print(f"\nEvaluando combinación: Recurso='{recurso}' | Tarea='{tarea}'")

        sublog = log_transformado[(log_transformado['org:resource'] == recurso) & (log_transformado['concept:name'] == tarea)]
        if sublog.empty:
            print("  - No hay datos para esta combinación, se omite.")
            continue

        # Cada fila de log_transformado guarda la lista de productividades del par (recurso, tarea)
        # en 'resource_productivity'. .values devolvería un array (1, dtype=object) que scipy no
        # puede pasar por np.isfinite, así que se desempaqueta la lista y se castea a float.
        datos = np.asarray(sublog['resource_productivity'].iloc[0], dtype=float)

        if len(datos) == 0:
            print(f"  - Sin muestras para el par ({recurso}, {tarea}), se omite.")
            continue
        
        #if np.all(datos == 0.0):
        #   print(f"  - Todas las muestras son 0.0 para el par ({recurso}, {tarea}), se omite.")
        #   continue

        #if len(datos) < 20:
            #print(f"  - Solo {len(datos)} muestras, puede no ser suficiente para un análisis robusto.")

        # Aquí llamaríamos a la función de búsqueda automática de distribuciones

        distribucion_ganadora = _encontrar_mejor_distribucion_AIC(datos, recurso, tarea)

        _representar_distribucion_ganadora(datos, distribucion_ganadora)
        
        RESULTADOS.append([recurso, tarea, distribucion_ganadora.name, nombre_log])

_representar_resultados(RESULTADOS)

In [ ]:
df_final = pd.DataFrame(RESULTADOS, columns=['Recurso', 'Tarea', 'Distribución Ganadora', 'nombre_log'])

print("\nRESULTADOS FINALES:")
print(df_final)

In [ ]:
# === Ground truth: distribución generadora real por (caso, recurso, tarea) ===
# Extraído de los prosimos_base.json en src/log_generation/resource_productivity/Caso_*.
GROUND_TRUTH = pd.DataFrame([
    # --- Caso A (5 tareas, 3 recursos con asignación parcial) ---
    {"caso": "Caso_A", "recurso": "Carlos", "tarea": "A", "real": "uniform"},
    {"caso": "Caso_A", "recurso": "Sandra", "tarea": "A", "real": "norm"},
    {"caso": "Caso_A", "recurso": "Carlos", "tarea": "B", "real": "expon"},
    {"caso": "Caso_A", "recurso": "Sandra", "tarea": "B", "real": "lognorm"},
    {"caso": "Caso_A", "recurso": "Carlos", "tarea": "C", "real": "norm"},
    {"caso": "Caso_A", "recurso": "Sandra", "tarea": "C", "real": "uniform"},
    {"caso": "Caso_A", "recurso": "Pedro",  "tarea": "D", "real": "lognorm"},
    {"caso": "Caso_A", "recurso": "Sandra", "tarea": "D", "real": "norm"},
    {"caso": "Caso_A", "recurso": "Pedro",  "tarea": "E", "real": "expon"},
    {"caso": "Caso_A", "recurso": "Sandra", "tarea": "E", "real": "uniform"},
    # --- Caso B (3 tareas, 6 recursos sin solapamiento, familias contrastadas por tarea) ---
    {"caso": "Caso_B", "recurso": "Carlos", "tarea": "A", "real": "expon"},
    {"caso": "Caso_B", "recurso": "Sandra", "tarea": "A", "real": "norm"},
    {"caso": "Caso_B", "recurso": "Pedro",  "tarea": "B", "real": "lognorm"},
    {"caso": "Caso_B", "recurso": "Laura",  "tarea": "B", "real": "uniform"},
    {"caso": "Caso_B", "recurso": "Marta",  "tarea": "C", "real": "gamma"},
    {"caso": "Caso_B", "recurso": "Diego",  "tarea": "C", "real": "norm"},
    # --- Caso C (5 tareas, 8 recursos con solapamiento entre tareas) ---
    {"caso": "Caso_C", "recurso": "Ana",    "tarea": "A", "real": "norm"},
    {"caso": "Caso_C", "recurso": "Bruno",  "tarea": "A", "real": "lognorm"},
    {"caso": "Caso_C", "recurso": "Carla",  "tarea": "A", "real": "expon"},
    {"caso": "Caso_C", "recurso": "Diego",  "tarea": "B", "real": "gamma"},
    {"caso": "Caso_C", "recurso": "Elena",  "tarea": "B", "real": "uniform"},
    {"caso": "Caso_C", "recurso": "Fer",    "tarea": "B", "real": "norm"},
    {"caso": "Caso_C", "recurso": "Ana",    "tarea": "C", "real": "expon"},
    {"caso": "Caso_C", "recurso": "Diego",  "tarea": "C", "real": "lognorm"},
    {"caso": "Caso_C", "recurso": "Gloria", "tarea": "C", "real": "gamma"},
    {"caso": "Caso_C", "recurso": "Bruno",  "tarea": "D", "real": "uniform"},
    {"caso": "Caso_C", "recurso": "Elena",  "tarea": "D", "real": "gamma"},
    {"caso": "Caso_C", "recurso": "Hugo",   "tarea": "D", "real": "norm"},
    {"caso": "Caso_C", "recurso": "Carla",  "tarea": "E", "real": "lognorm"},
    {"caso": "Caso_C", "recurso": "Fer",    "tarea": "E", "real": "expon"},
    {"caso": "Caso_C", "recurso": "Hugo",   "tarea": "E", "real": "norm"},
    # --- Caso D (5 tareas, 3 recursos asignados a todas, misma familia por tarea) ---
    {"caso": "Caso_D", "recurso": "Carlos", "tarea": "A", "real": "norm"},
    {"caso": "Caso_D", "recurso": "Sandra", "tarea": "A", "real": "norm"},
    {"caso": "Caso_D", "recurso": "Pedro",  "tarea": "A", "real": "norm"},
    {"caso": "Caso_D", "recurso": "Carlos", "tarea": "B", "real": "lognorm"},
    {"caso": "Caso_D", "recurso": "Sandra", "tarea": "B", "real": "lognorm"},
    {"caso": "Caso_D", "recurso": "Pedro",  "tarea": "B", "real": "lognorm"},
    {"caso": "Caso_D", "recurso": "Carlos", "tarea": "C", "real": "gamma"},
    {"caso": "Caso_D", "recurso": "Sandra", "tarea": "C", "real": "gamma"},
    {"caso": "Caso_D", "recurso": "Pedro",  "tarea": "C", "real": "gamma"},
    {"caso": "Caso_D", "recurso": "Carlos", "tarea": "D", "real": "expon"},
    {"caso": "Caso_D", "recurso": "Sandra", "tarea": "D", "real": "expon"},
    {"caso": "Caso_D", "recurso": "Pedro",  "tarea": "D", "real": "expon"},
    {"caso": "Caso_D", "recurso": "Carlos", "tarea": "E", "real": "uniform"},
    {"caso": "Caso_D", "recurso": "Sandra", "tarea": "E", "real": "uniform"},
    {"caso": "Caso_D", "recurso": "Pedro",  "tarea": "E", "real": "uniform"},
])

# === Preparar resultados predichos a partir de df_final ===
# Normalizamos nombres de columnas y extraemos el identificador de caso del nombre del log.
df_predicho = df_final.rename(columns={
    'Recurso': 'recurso',
    'Tarea': 'tarea',
    'Distribución Ganadora': 'ganadora'
}).copy()
df_predicho['caso']      = df_predicho['nombre_log'].str.extract(r'Caso_([A-D])')[0].apply(lambda c: f'Caso_{c}')
df_predicho['log_corto'] = df_predicho['nombre_log'].str.extract(r'(\d+)\.csv')[0]

# === Cruce con ground truth ===
df_comparacion = df_predicho.merge(GROUND_TRUTH, on=['caso', 'recurso', 'tarea'], how='left')
df_comparacion['acierto'] = df_comparacion['ganadora'] == df_comparacion['real']

# === Resumen global ===
total    = len(df_comparacion)
aciertos = int(df_comparacion['acierto'].sum())
print(f"ACIERTO GLOBAL: {aciertos}/{total} ({aciertos/total*100:.1f}%)\n")

# === Aciertos por caso y tamaño (para ver el efecto del volumen) ===
print("Aciertos por caso y tamaño:")
print(
    df_comparacion.groupby(['caso', 'log_corto'])['acierto']
                  .agg(aciertos='sum', total='count')
                  .assign(porcentaje=lambda df: (df['aciertos'] / df['total'] * 100).round(1))
                  .to_string()
)

# === Aciertos por familia generadora (qué familias se identifican peor) ===
print("\nAciertos por familia generadora:")
print(
    df_comparacion.groupby('real')['acierto']
                  .agg(aciertos='sum', total='count')
                  .assign(porcentaje=lambda df: (df['aciertos'] / df['total'] * 100).round(1))
                  .sort_values('porcentaje', ascending=False)
                  .to_string()
)

# === Matriz de confusión (qué tiende a confundir con qué) ===
print("\nMatriz de confusión (filas = familia generadora, columnas = familia ganadora):")
print(pd.crosstab(df_comparacion['real'], df_comparacion['ganadora'], margins=True))

# === Detalle nominal de errores ===
errores = df_comparacion[~df_comparacion['acierto']][['caso', 'log_corto', 'recurso', 'tarea', 'real', 'ganadora']]
if not errores.empty:
    print("\nDetalle de errores:")
    print(errores.to_string(index=False))
else:
    print("\nSin errores.")


## Comparativa AIC vs KS

In [ ]:
# === Barrido con el criterio KS ===
# Mismo pipeline que el bucle AIC de más arriba, pero seleccionando la familia por la
# distancia máxima D del test KS en lugar de por AIC. No se generan las gráficas por
# combinación.
RESULTADOS_KS = []

for nombre_log in LISTADO_LOGS:
    # Reset del calendario singleton para que cada log use el suyo (ver nota en el bucle AIC).
    _rp._obtener_calendario.calendario_estatico = None

    log = _cargar_log(nombre_log)

    log_filtrado = filter_resource_productivity(log, parametros={'debug': False})
    log_transformado = transformacion_resource_productivity(log_filtrado, parametros={'debug': False})

    recursos = log_transformado['org:resource'].unique()
    tareas = log_transformado['concept:name'].unique()
    combinaciones = list(itertools.product(recursos, tareas))

    for recurso, tarea in combinaciones:

        sublog = log_transformado[(log_transformado['org:resource'] == recurso) & (log_transformado['concept:name'] == tarea)]
        if sublog.empty:
            continue

        datos = np.asarray(sublog['resource_productivity'].iloc[0], dtype=float)
        if len(datos) == 0:
            continue

        distribucion_ganadora = _encontrar_mejor_distribucion_KS(datos, recurso, tarea)
        if distribucion_ganadora is None:
            continue

        RESULTADOS_KS.append([recurso, tarea, distribucion_ganadora.name, nombre_log])

df_ks = pd.DataFrame(RESULTADOS_KS, columns=['Recurso', 'Tarea', 'Distribución Ganadora', 'nombre_log'])
print("\nRESULTADOS KS:")
print(df_ks)


In [ ]:
# === Comparativa AIC vs KS ===
# El criterio AIC ya quedó evaluado en `df_comparacion` (celda anterior). Evaluamos el
# KS del mismo modo contra el ground truth y comparamos ambos criterios.
df_predicho_ks = df_ks.rename(columns={
    'Recurso': 'recurso', 'Tarea': 'tarea', 'Distribución Ganadora': 'ganadora'
}).copy()
df_predicho_ks['caso']      = df_predicho_ks['nombre_log'].str.extract(r'Caso_([A-D])')[0].apply(lambda c: f'Caso_{c}')
df_predicho_ks['log_corto'] = df_predicho_ks['nombre_log'].str.extract(r'(\d+)\.csv')[0]
df_comparacion_ks = df_predicho_ks.merge(GROUND_TRUTH, on=['caso', 'recurso', 'tarea'], how='left')
df_comparacion_ks['acierto'] = df_comparacion_ks['ganadora'] == df_comparacion_ks['real']

# --- Acierto global ---
total = len(df_comparacion)
g_aic = df_comparacion['acierto'].mean() * 100
g_ks  = df_comparacion_ks['acierto'].mean() * 100
print(f"ACIERTO GLOBAL AIC: {int(df_comparacion['acierto'].sum())}/{total} ({g_aic:.1f}%)")
print(f"ACIERTO GLOBAL KS : {int(df_comparacion_ks['acierto'].sum())}/{total} ({g_ks:.1f}%)")
print(f"Ventaja AIC: {g_aic - g_ks:+.1f} puntos porcentuales\n")

# --- Acierto por familia generadora ---
ORDEN_FAMILIAS = ['norm', 'uniform', 'gamma', 'expon', 'lognorm']
fam_aic = df_comparacion.groupby('real')['acierto'].mean() * 100
fam_ks  = df_comparacion_ks.groupby('real')['acierto'].mean() * 100
tabla_familia = pd.DataFrame({'AIC (%)': fam_aic.round(1), 'KS (%)': fam_ks.round(1)})
tabla_familia['Δ (AIC-KS)'] = (tabla_familia['AIC (%)'] - tabla_familia['KS (%)']).round(1)
tabla_familia = tabla_familia.reindex(ORDEN_FAMILIAS)
print("Acierto por familia generadora:")
print(tabla_familia.to_string())

# --- Figura 1: matrices de confusión lado a lado ---
# Anotaciones dibujadas a mano (sns.heatmap dejaba sin número algunas celdas según versión).
FAMILIAS_ORD = ['expon', 'gamma', 'lognorm', 'norm', 'uniform']
fig, ejes = plt.subplots(1, 2, figsize=(15, 6))
for eje, comp, titulo in zip(ejes, [df_comparacion, df_comparacion_ks], ['AIC', 'KS (distancia D)']):
    cm = pd.crosstab(comp['real'], comp['ganadora']).reindex(
        index=FAMILIAS_ORD, columns=FAMILIAS_ORD, fill_value=0
    ).to_numpy()
    eje.imshow(cm, cmap='Blues', vmin=0)
    umbral = cm.max() / 2.0  # contraste: blanco en celdas oscuras, gris en claras
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            eje.text(j, i, int(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > umbral else '#333333', fontsize=12)
    eje.set_xticks(range(len(FAMILIAS_ORD)))
    eje.set_yticks(range(len(FAMILIAS_ORD)))
    eje.set_xticklabels(FAMILIAS_ORD)
    eje.set_yticklabels(FAMILIAS_ORD)
    eje.set_xticks(np.arange(-0.5, len(FAMILIAS_ORD), 1), minor=True)
    eje.set_yticks(np.arange(-0.5, len(FAMILIAS_ORD), 1), minor=True)
    eje.grid(which='minor', color='white', linewidth=1.5)
    eje.tick_params(which='minor', length=0)
    eje.set_title(f"{titulo} — acierto {comp['acierto'].mean() * 100:.1f}%")
    eje.set_xlabel('Familia predicha')
    eje.set_ylabel('Familia generadora real')
fig.suptitle('Matrices de confusión por criterio de selección', fontsize=14)
plt.tight_layout()
plt.show()

# --- Figura 2: acierto por familia (barras agrupadas) ---
etiquetas    = ORDEN_FAMILIAS + ['GLOBAL']
valores_aic  = [fam_aic.get(f, np.nan) for f in ORDEN_FAMILIAS] + [g_aic]
valores_ks   = [fam_ks.get(f, np.nan) for f in ORDEN_FAMILIAS] + [g_ks]
x = np.arange(len(etiquetas))
ancho = 0.38
fig, ax = plt.subplots(figsize=(11, 6))
barras_aic = ax.bar(x - ancho / 2, valores_aic, ancho, label='AIC', color='#2ca02c')
barras_ks  = ax.bar(x + ancho / 2, valores_ks, ancho, label='KS (distancia D)', color='#d62728')
ax.set_xticks(x)
ax.set_xticklabels(etiquetas)
ax.set_ylabel('Acierto (%)')
ax.set_ylim(0, 112)
ax.set_title('Identificación de la familia generadora: AIC vs KS')
ax.bar_label(barras_aic, fmt='%.0f', padding=2)
ax.bar_label(barras_ks, fmt='%.0f', padding=2)
ax.axvline(len(ORDEN_FAMILIAS) - 0.5, color='grey', ls='--', lw=0.9)
ax.legend(title='Criterio de selección')
plt.tight_layout()
plt.show()
